# M2.2.e: 169-feature integration + LightGBM val AUC

Integrates all M2.2 sub-steps into a single pipeline and runs LightGBM.

Pipeline: cloud_coverage fix → ClusterNo merge → sort → value-change (120)
→ SavGol residual → dayofyear → downsampling → CV split → StandardScaler → LightGBM

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from scipy.signal import savgol_filter
import lightgbm as lgb

train = pd.read_csv("../data/raw/train_features.csv")
print(f"Loaded: {train.shape}")

# M2.2.0: cloud_coverage sentinel fix
train["cloud_coverage"] = train["cloud_coverage"].replace({255: 10})
print("cloud_coverage fix applied")

In [ ]:
clusterno_df = pd.read_csv("../data/interim/clusterno.csv")
train = train.merge(clusterno_df, on="building_id", how="left")
nan_clusterno = train["ClusterNo"].isna().sum()
print(f"After ClusterNo merge: {train.shape}")
print(f"ClusterNo NaN: {nan_clusterno}")
assert nan_clusterno == 0, f"Unexpected ClusterNo NaN: {nan_clusterno}"

In [ ]:
train = train.sort_values(["building_id", "timestamp"]).reset_index(drop=True)

shifts = (
    list(np.arange(-24, 0))
    + list(np.arange(1, 25))
    + list(np.arange(-168, -24, 24))
    + list(np.arange(48, 169, 24))
)

for n in shifts:
    train[f"lag_value_{n}"] = (
        train.groupby("building_id")["meter_reading"].shift(n) - train["meter_reading"]
    )
    train[f"lag_value_ratio_{n}"] = (
        train.groupby("building_id")["meter_reading"].shift(n) + 1
    ) / (train["meter_reading"] + 1)

lag_cols = [c for c in train.columns if c.startswith("lag_value_")]
print(f"Lag features: {len(lag_cols)}")
print(f"Train shape: {train.shape}")
assert len(lag_cols) == 120, f"Expected 120, got {len(lag_cols)}"

In [ ]:
results = []
for bid in train["building_id"].unique():
    tmp = train[train["building_id"] == bid].copy()
    smoothed = savgol_filter(tmp["meter_reading"].ffill().bfill().fillna(0), 5, 3)
    tmp["Residual_savgol_w5p3"] = tmp["meter_reading"] - smoothed
    results.append(tmp)
train = pd.concat(results).sort_index()
print(f"After SavGol: {train.shape}")

In [ ]:
train["dayofyear"] = (
    pd.to_datetime(train["timestamp"]).dt.dayofyear
    + pd.to_datetime(train["timestamp"]).dt.hour / 24
)
print(f"After dayofyear: {train.shape}")

In [ ]:
X_full = train.select_dtypes(
    include=["float", "int", "int64", "int32", "float64", "float32"]
)
drop_cols = ["anomaly", "wind_direction", "air_temperature_std_lag73"]
X_full = X_full.drop(columns=[c for c in drop_cols if c in X_full.columns])

print(f"Feature count: {X_full.shape[1]}")
print("Expected: 169")
assert X_full.shape[1] == 169, (
    f"Feature count mismatch: got {X_full.shape[1]}, expected 169"
)
print("Feature count = 169")

categories = {
    "raw_numeric": 0,
    "lag_value_diff": 0,
    "lag_value_ratio": 0,
    "ClusterNo": 0,
    "Residual_savgol_w5p3": 0,
    "dayofyear": 0,
}
for col in X_full.columns:
    if col.startswith("lag_value_ratio_"):
        categories["lag_value_ratio"] += 1
    elif col.startswith("lag_value_"):
        categories["lag_value_diff"] += 1
    elif col == "ClusterNo":
        categories["ClusterNo"] += 1
    elif col == "Residual_savgol_w5p3":
        categories["Residual_savgol_w5p3"] += 1
    elif col == "dayofyear":
        categories["dayofyear"] += 1
    else:
        categories["raw_numeric"] += 1

print()
print("Feature category breakdown:")
for cat, count in categories.items():
    print(f"  {cat}: {count}")
print(f"  Total: {sum(categories.values())}")

In [ ]:
print("NaN rate per feature class:")
class_specs = [
    (
        "raw_numeric",
        lambda c: not c.startswith("lag_value_")
        and c not in ["ClusterNo", "Residual_savgol_w5p3", "dayofyear"],
    ),
    (
        "lag_value_diff",
        lambda c: c.startswith("lag_value_") and not c.startswith("lag_value_ratio_"),
    ),
    ("lag_value_ratio", lambda c: c.startswith("lag_value_ratio_")),
    ("ClusterNo", lambda c: c == "ClusterNo"),
    ("Residual_savgol_w5p3", lambda c: c == "Residual_savgol_w5p3"),
    ("dayofyear", lambda c: c == "dayofyear"),
]
for name, fn in class_specs:
    cols = [c for c in X_full.columns if fn(c)]
    if not cols:
        continue
    total = len(cols) * len(X_full)
    nans = X_full[cols].isna().sum().sum()
    print(f"  {name:25s} {len(cols):4d} cols  NaN: {nans / total * 100:5.2f}%")

In [ ]:
neg = train[train["anomaly"] == 0]
pos = train[train["anomaly"] == 1]
print(f"Pre-downsampling: {len(neg):,} normal + {len(pos):,} anomaly")

negs1 = neg.sample(n=pos.shape[0], random_state=10)
negs2 = neg.sample(n=pos.shape[0], random_state=20)
df_eq = pd.concat([negs1, pos, negs2, pos], axis=0).reset_index(drop=True)

print(f"Post-downsampling: {df_eq.shape[0]:,} rows")
print(
    f"Class balance: {(df_eq['anomaly'] == 0).sum():,} : {(df_eq['anomaly'] == 1).sum():,}"
)

In [ ]:
X_eq = df_eq.select_dtypes(include=["float", "int"])
X_eq = X_eq.drop(columns=[c for c in drop_cols if c in X_eq.columns])
print(f"X_eq features: {X_eq.shape[1]} (expected 169)")
assert X_eq.shape[1] == 169

train_mask = df_eq["building_id"] % 5 < 4
val_mask = df_eq["building_id"] % 5 == 4

X_train = X_eq[train_mask]
y_train = df_eq.loc[train_mask, "anomaly"]
X_val = X_eq[val_mask]
y_val = df_eq.loc[val_mask, "anomaly"]

print(f"Train: {X_train.shape}, {df_eq[train_mask]['building_id'].nunique()} buildings")
print(f"Val: {X_val.shape}, {df_eq[val_mask]['building_id'].nunique()} buildings")

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
print(f"X_train_scaled: {X_train_scaled.shape}, NaN={np.isnan(X_train_scaled).sum():,}")
print(f"X_val_scaled:   {X_val_scaled.shape}, NaN={np.isnan(X_val_scaled).sum():,}")

In [ ]:
model = lgb.LGBMClassifier(n_estimators=100, verbose=-1)
model.fit(X_train_scaled, y_train)

preds = model.predict_proba(X_val_scaled)[:, 1]
val_auc = roc_auc_score(y_val, preds)

print(f"{'=' * 60}")
print(f"M2.2.e LightGBM val AUC: {val_auc:.4f}")
print(f"{'=' * 60}")

In [ ]:
baseline_auc = 0.8952  # M2.1
paper_target = 0.9849  # paper Table 2
delta_auc = val_auc - baseline_auc

print(f"M2.1 baseline (57 features):    {baseline_auc:.4f}")
print(f"M2.2.e (169 features):          {val_auc:.4f}")
print(f"delta_AUC vs M2.1:              {delta_auc:+.4f}")
print(f"Paper Table 2 target:           {paper_target:.4f}")
print(f"Gap vs paper:                   {(paper_target - val_auc) * 100:+.2f}%")
print()
print("Done when criteria:")
print(f"  val AUC >= 0.97:  {'pass' if val_auc >= 0.97 else 'FAIL'}")
print(f"  delta_AUC >= +0.04: {'pass' if delta_auc >= 0.04 else 'FAIL'}")

In [ ]:
importances = pd.Series(model.feature_importances_, index=X_train.columns)
top10 = importances.nlargest(10)

print("Our top 10 importance:")
for i, (feat, imp) in enumerate(top10.items(), 1):
    print(f"  {i:2d}. {feat:35s} {imp:6.0f}")

paper_top10 = [
    "building_id",
    "lag_value_ratio_1",
    "lag_value_ratio_-1",
    "meter_reading",
    "dayofyear",
    "square_feet",
    "gte_building_id",
    "lag_value_ratio_-168",
    "lag_value_ratio_2",
    "gte_meter_primary_use",
]
print()
print("Paper Fig 5 top 10 (our name mapping):")
for i, feat in enumerate(paper_top10, 1):
    marker = "in ours" if feat in top10.index else "-"
    print(f"  {i:2d}. {feat:35s} {marker}")

overlap = set(paper_top10) & set(top10.index)
print(f"Overlap: {len(overlap)}/10")